# Recommender Systems Demo: Item-Based Collaborative Filtering (IB-CF)

This notebook demonstrates **Item-Based Collaborative Filtering (IB-CF)** using a small user–item rating matrix.

**You will learn**
- How to compute **item–item similarity** (cosine / correlation options)
- How to predict a missing rating using a **similarity-weighted average**
- How to produce **Top‑N recommendations** for a target user

> No external downloads — the notebook is self-contained and designed for classroom demos.


In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.precision", 3)
np.set_printoptions(precision=3, suppress=True)

<Token var=<ContextVar name='format_options' default={'edgeitems': 3, 'threshold': 1000, 'floatmode': 'maxprec', 'precision': 8, 'suppress': False, 'linewidth': 75, 'nanstr': 'nan', 'infstr': 'inf', 'sign': '-', 'formatter': None, 'legacy': 9223372036854775807, 'override_repr': None} at 0x103a6dda0> at 0x110f4ce40>

## 1) Create a small user–item rating matrix

- Rows are users, columns are items (movies).
- Missing ratings are represented as `NaN`.


In [2]:
ratings = pd.DataFrame(
    {
        "M1": [5, 4, 5, 1, np.nan],
        "M2": [2, 2, 3, 5, 4],
        "M3": [np.nan, 5, 4, 1, 4],
        "M4": [1, 1, 2, 4, np.nan],
    },
    index=["U1", "U2", "U3", "U4", "U5"]
)
ratings

,M1,M2,M3,M4
U1,5.0,2,NaN,1.0
U2,4.0,2,5.0,1.0
U3,5.0,3,4.0,2.0
U4,1.0,5,1.0,4.0
U5,NaN,4,4.0,NaN


## 2) Item–Item similarity functions

We implement:
- **Cosine similarity** (on co-rated users)
- **Pearson correlation** (item-centered)
- **Adjusted cosine** (user-centered; reduces user bias)

For intro lectures, cosine is simplest. In practice, adjusted cosine is often more robust.


In [3]:
def co_rated_vectors(ratings_df: pd.DataFrame, item_a: str, item_b: str):
    """Aligned rating vectors for users who rated both items."""
    a = ratings_df[item_a]
    b = ratings_df[item_b]
    mask = a.notna() & b.notna()
    return a[mask].to_numpy(dtype=float), b[mask].to_numpy(dtype=float)

def cosine_sim_items(ratings_df: pd.DataFrame, item_a: str, item_b: str) -> float:
    x, y = co_rated_vectors(ratings_df, item_a, item_b)
    if len(x) == 0:
        return np.nan
    num = np.dot(x, y)
    den = np.linalg.norm(x) * np.linalg.norm(y)
    return float(num / den) if den > 0 else np.nan

def pearson_sim_items(ratings_df: pd.DataFrame, item_a: str, item_b: str) -> float:
    x, y = co_rated_vectors(ratings_df, item_a, item_b)
    if len(x) < 2:
        return np.nan
    x = x - x.mean()
    y = y - y.mean()
    num = np.dot(x, y)
    den = np.linalg.norm(x) * np.linalg.norm(y)
    return float(num / den) if den > 0 else np.nan

def adjusted_cosine_sim_items(ratings_df: pd.DataFrame, item_a: str, item_b: str) -> float:
    a = ratings_df[item_a]
    b = ratings_df[item_b]
    user_means = ratings_df.mean(axis=1, skipna=True)
    mask = a.notna() & b.notna() & user_means.notna()
    if mask.sum() < 2:
        return np.nan
    x = (a[mask] - user_means[mask]).to_numpy(dtype=float)
    y = (b[mask] - user_means[mask]).to_numpy(dtype=float)
    num = np.dot(x, y)
    den = np.linalg.norm(x) * np.linalg.norm(y)
    return float(num / den) if den > 0 else np.nan

## 3) Compute an item–item similarity matrix (Cosine)

We compute a full item–item matrix using cosine similarity on co-rated users.


In [4]:
items = ratings.columns.tolist()

sim_cos = pd.DataFrame(index=items, columns=items, dtype=float)
for i in items:
    for j in items:
        sim_cos.loc[i, j] = cosine_sim_items(ratings, i, j)

sim_cos

,M1,M2,M3,M4
M1,1.000,0.716,0.976,0.599
M2,0.716,1.000,0.768,0.987
M3,0.976,0.768,1.000,0.572
M4,0.599,0.987,0.572,1.000


## 4) Find the Top‑K similar items to a target item

Example: items most similar to **M3**.


In [5]:
target_item = "M3"
sim_cos[target_item].sort_values(ascending=False)

M3    1.000
M1    0.976
M2    0.768
M4    0.572
Name: M3, dtype: float64

## 5) Predict a missing rating with Item‑Based CF (IB‑CF)

For a target user `u` and target item `i`:
1. Look at items the user has already rated  
2. Keep the **Top‑K** most similar to `i`  
3. Predict using a **similarity‑weighted average**

We use `abs(sim)` in the denominator to avoid sign cancellation.


In [6]:
def predict_ibcf(
    ratings_df: pd.DataFrame,
    user: str,
    item: str,
    k: int = 2,
    sim_matrix: pd.DataFrame | None = None
) -> float:
    if sim_matrix is None:
        raise ValueError("Provide a precomputed item–item similarity matrix.")

    user_ratings = ratings_df.loc[user].drop(labels=[item], errors="ignore")
    rated_items = user_ratings[user_ratings.notna()]

    if rated_items.empty:
        return np.nan

    sims = sim_matrix.loc[rated_items.index, item].dropna()
    if sims.empty:
        return np.nan

    sims = sims.reindex(sims.abs().sort_values(ascending=False).head(k).index)

    num = float(np.sum(sims.to_numpy() * rated_items.loc[sims.index].to_numpy()))
    den = float(np.sum(np.abs(sims.to_numpy())))
    return num / den if den > 0 else np.nan

### Example: Predict **U1**'s rating for **M3** (currently missing)

In [7]:
pred_u1_m3 = predict_ibcf(ratings, user="U1", item="M3", k=2, sim_matrix=sim_cos)
pred_u1_m3

3.6787090773369653

## 6) Explain the neighbors used in the prediction

This supports “because you liked …” explanations.


In [8]:
def explain_prediction(ratings_df, user, item, k, sim_matrix):
    user_ratings = ratings_df.loc[user].drop(labels=[item], errors="ignore")
    rated_items = user_ratings[user_ratings.notna()]
    sims = sim_matrix.loc[rated_items.index, item].dropna()
    sims = sims.reindex(sims.abs().sort_values(ascending=False).head(k).index)

    out = pd.DataFrame({
        "rated_item": sims.index,
        "sim_to_target": sims.values,
        "user_rating": rated_items.loc[sims.index].values,
        "contribution": sims.values * rated_items.loc[sims.index].values
    }).reset_index(drop=True)
    return out

explain_prediction(ratings, "U1", "M3", k=2, sim_matrix=sim_cos)

,rated_item,sim_to_target,user_rating,contribution
0,M1,0.976,5.0,4.881
1,M2,0.768,2.0,1.537


## 7) Top‑N recommendations for a user

Predict ratings for all **unrated** items and recommend the highest ones.


In [9]:
def recommend_top_n(ratings_df, user, n=3, k=2, sim_matrix=None):
    unrated = ratings_df.columns[ratings_df.loc[user].isna()]
    preds = []
    for item in unrated:
        p = predict_ibcf(ratings_df, user, item, k=k, sim_matrix=sim_matrix)
        preds.append((item, p))
    recs = pd.DataFrame(preds, columns=["item", "pred_rating"]).sort_values("pred_rating", ascending=False)
    return recs.head(n)

recommend_top_n(ratings, "U1", n=5, k=2, sim_matrix=sim_cos)

,item,pred_rating
0,M3,3.679


## 8) Notes / Extensions

For real systems:
- Compute item–item similarities **offline**, store only Top‑K neighbors per item.
- Try **adjusted cosine** or **Pearson** to handle user/item bias.
- For extreme sparsity, consider **matrix factorization** methods.
